In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader, random_split
from torchvision.models import ResNet50_Weights
from torchsummary import summary
from torchvision import datasets, transforms, models

import os

In [2]:
epochs = 15
batch_size = 32
learning_rate = 0.001

In [3]:
# data preparation

data_transform = transforms.Compose([

        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

])

data_dir = "/content/drive/MyDrive/ALL DATASET/rooms_dataset"

In [4]:
dataset = datasets.ImageFolder(
    root = data_dir,
    transform = data_transform
)

In [5]:
# Check class names

print("Classes:", dataset.classes)


# Number of classes

num_classes = len(dataset.classes)
print("Number of classes:", num_classes)

Classes: ['bed_room', 'dining_room', 'living_room']
Number of classes: 3


In [6]:
# Train Validation Split

train_size = int(0.8* len(dataset))
test_size  = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])


print("Training images:", len(train_dataset))
print("Validation images:", len(test_dataset))

Training images: 94
Validation images: 24


In [7]:
# Data Loader

train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

In [9]:
# Load ResNet50 Model
model = models.resnet50(weights = ResNet50_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 102MB/s] 


In [10]:
model.fc = nn.Linear(
    model.fc.in_features, num_classes
)

In [11]:
# Freeze ResNet Layers
for params in model.parameters():
  params.requires_grad = False

In [12]:
# train only classification layer

for params in model.fc.parameters():
  params.requires_grad = True

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [14]:
criterion = nn.CrossEntropyLoss()


optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

In [17]:
# trainng function

def training_model(model, criterion, optimizer, train_loader, test_loader, epochs):
  for epoch in range(epochs):
    model.train()
    train_correct = 0
    train_total  = 0
    running_loss = 0

    print(f"\n Epoch {epoch+1}/ {epochs}")
    print("="*30)

    for batch, (images, labels) in enumerate(train_loader):
      images = images.to(device)
      labels =labels.to(device)

      # remove old gradient
      optimizer.zero_grad()

      #forward pass
      outputs = model(images)

      # calculate loss
      loss = criterion(outputs, labels)

      # backward pass
      loss.backward()

      # parameters update
      optimizer.step()

      running_loss += loss.item()

      _, predicted  = torch.max(outputs, 1)

      train_total += labels.size(0)

      train_correct += (predicted == labels).sum().item()


      if (batch+1) % 10 == 0:
         print(f"Batch {batch+1}/{len(train_loader)} Loss: {loss.item():.4f}")


    train_accuracy = (train_correct / train_total)*100


    print(f"Training Loss: {running_loss:.4f}")

    print( f"Training Accuracy: {train_accuracy:.4f}")


    # validation


    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
      for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        val_total += labels.size(0)
        val_correct += (predicted == labels).sum().item()

    val_accuracy = (val_correct / val_total)*100

    print(f"Validation Accuracy: {val_accuracy:.4f}")




In [18]:
# training model

training_model(model, criterion, optimizer, train_loader, test_loader, epochs)


 Epoch 1/ 15
Training Loss: 3.1658
Training Accuracy: 45.7447
Validation Accuracy: 37.5000

 Epoch 2/ 15
Training Loss: 2.8250
Training Accuracy: 67.0213
Validation Accuracy: 50.0000

 Epoch 3/ 15
Training Loss: 2.5280
Training Accuracy: 74.4681
Validation Accuracy: 66.6667

 Epoch 4/ 15
Training Loss: 2.3285
Training Accuracy: 78.7234
Validation Accuracy: 66.6667

 Epoch 5/ 15
Training Loss: 2.1249
Training Accuracy: 88.2979
Validation Accuracy: 75.0000

 Epoch 6/ 15
Training Loss: 1.9082
Training Accuracy: 91.4894
Validation Accuracy: 79.1667

 Epoch 7/ 15
Training Loss: 1.8284
Training Accuracy: 94.6809
Validation Accuracy: 87.5000

 Epoch 8/ 15
Training Loss: 1.6762
Training Accuracy: 94.6809
Validation Accuracy: 83.3333

 Epoch 9/ 15
Training Loss: 1.5291
Training Accuracy: 96.8085
Validation Accuracy: 79.1667

 Epoch 10/ 15
Training Loss: 1.3790
Training Accuracy: 96.8085
Validation Accuracy: 87.5000

 Epoch 11/ 15
Training Loss: 1.2504
Training Accuracy: 96.8085
Validation Accu